In [2]:
import numpy as np
import rasterio
import geopandas as gpd
import matplotlib.pyplot as plt

from scipy.ndimage import binary_opening, binary_closing
from skimage.measure import label, regionprops
from rasterio.features import shapes
from shapely.geometry import shape
from rasterio.features import shapes

from sqlalchemy import create_engine

# ============================================================
# 1. FEATURE EXTRACTION
# ============================================================
def compute_features(img):
    """
    Expected band order:
    [Blue, Green, Red, RedEdge, NIR, SWIR1, SWIR2]

    Output shape:
    (H, W, 9)
    """
    with rasterio.open(img) as src:
        blue = src.read(1).astype("float32")
        green = src.read(2).astype("float32")
        red = src.read(3).astype("float32")
        re1 = src.read(4).astype("float32")
        nir = src.read(7).astype("float32")
        swir1 = src.read(9).astype("float32")
        swir2 = src.read(10).astype("float32")

    # Normalize reflectance
    bluef  = blue / 10000
    greenf = green / 10000
    redf   = red / 10000
    re1f   = re1 / 10000
    nirf   = nir / 10000
    swir1f = swir1 / 10000
    swir2f = swir2 / 10000

    eps = 1e-9

    # Spectral indices
    ndvi = (nirf - redf) / (nirf + redf + eps)
    ndre = (nirf - re1f) / (nirf + re1f + eps)
    ndmi = (nirf - swir1f) / (nirf + swir1f + eps)
    nbr  = (nirf - swir2f) / (nirf + swir2f + eps)
    evi  = 2.5 * (nirf - redf) / (nirf + 6*redf - 7.5*bluef + 1 + eps)

    # Floating Algae Index (you used this in training)
    lambda_red = 665
    lambda_nir = 842
    lambda_swir1 = 1610

    nir_baseline = redf + (swir1f - redf) * (
        (lambda_nir - lambda_red) / (lambda_swir1 - lambda_red)
    )
    fai = nirf - nir_baseline

    # Final feature stack for RF
    feature_stack = np.stack(
        [re1f, nirf, swir1f],
        axis=-1
    )
    # print("Feature stack shape:", feature_stack.shape)
    # print("Feature mins:", np.min(feature_stack, axis=(0,1)))
    # print("Feature maxs:", np.max(feature_stack, axis=(0,1)))
    return feature_stack


# ============================================================
# 2. RF PREDICTION
# ============================================================
def predict_vegetation_mask(tif_path, model):
    with rasterio.open(tif_path) as src:
        img = src.read()

    features = compute_features(tif_path)
    h, w, n_features = features.shape

    # reshape correctly
    X = features.reshape(-1, n_features)
    
    # VALID PIXELS ONLY (same as your manual code)
    valid = np.all(np.isfinite(X), axis=1)
    # print("Total pixels:", X.shape[0])
    # print("Valid pixels:", np.sum(valid))
    # initialize prediction
    pred_flat = np.zeros(X.shape[0], dtype=np.uint8)

    # predict ONLY valid pixels
    pred_flat[valid] = model.predict(X)

    # reshape back
    mask = pred_flat.reshape(h, w)

    return mask, img


# ============================================================
# 3. RAW DEFORESTATION
# ============================================================
def get_raw_deforestation(before_mask, after_mask):
    """
    Deforestation = vegetation before AND not vegetation after
    """
    return ((before_mask == 1) & (after_mask == 0)).astype(np.uint8)


# ============================================================
# 4. CLEAN MASK
# ============================================================
def clean_deforestation_mask(raw_mask, open_size=3, close_size=5):
    clean = binary_opening(raw_mask, structure=np.ones((open_size, open_size)))
    clean = binary_closing(clean, structure=np.ones((close_size, close_size)))
    return clean.astype(np.uint8)


# ============================================================
# 5. REMOVE SMALL PATCHES
# ============================================================
def remove_small_patches(mask, min_patch_pixels=100):
    labeled = label(mask)
    regions = regionprops(labeled)

    filtered = np.zeros_like(mask, dtype=np.uint8)

    for region in regions:
        if region.area >= min_patch_pixels:
            filtered[labeled == region.label] = 1

    return filtered


def save_mask_as_tif(reference_path, mask, output_path):
    with rasterio.open(reference_path) as src:
        profile = src.profile.copy()

        profile.update(
            dtype=rasterio.uint8,
            count=1,
            compress='lzw',
            nodata=0
        )

        mask_to_save = (mask.astype(np.uint8) * 255)

        with rasterio.open(output_path, 'w', **profile) as dst:
            dst.write(mask_to_save, 1)



# ============================================================
# 6. MAIN DEBUG FUNCTION
# ============================================================
def detect_deforestation(
    before_path,
    after_path,
    model,
    min_patch_pixels=100,
    open_size=3,
    close_size=5,
):
    """
    Detect deforestation between before and after TIFFs.

    For now:
    - NO polygon conversion
    - NO saving
    - ONLY visualization + masks

    Returns:
    - before_mask
    - after_mask
    - raw_deforestation
    - filtered_mask
    """

    # Step 1: Predict vegetation masks
    before_mask, before_img = predict_vegetation_mask(before_path, model)
    after_mask, after_img = predict_vegetation_mask(after_path, model)

    # Step 2: Detect raw vegetation loss
    raw_deforestation = get_raw_deforestation(before_mask, after_mask)

    # Step 3: Clean noise
    clean_mask = clean_deforestation_mask(raw_deforestation, open_size, close_size)

    # Step 4: Remove tiny patches
    filtered_mask = remove_small_patches(clean_mask, min_patch_pixels=min_patch_pixels)
    # Step 6: Save outputs
    save_mask_as_tif(before_path, raw_deforestation, "raw_deforestation.tif")
    save_mask_as_tif(before_path, filtered_mask, "clean_deforestation.tif")
    # Step 5: Visualize
    """
    if visualize:
        before_rgb = np.dstack([before_img[2], before_img[1], before_img[0]])
        after_rgb  = np.dstack([after_img[2], after_img[1], after_img[0]])

        # simple contrast stretch
        before_rgb = before_rgb / np.percentile(before_rgb, 98)
        after_rgb  = after_rgb / np.percentile(after_rgb, 98)

        before_rgb = np.clip(before_rgb, 0, 1)
        after_rgb  = np.clip(after_rgb, 0, 1)

        fig, axes = plt.subplots(2, 3, figsize=(18, 10))
        
    
        axes[0, 0].imshow(before_rgb)
        axes[0, 0].set_title("Before RGB")
        axes[0, 0].axis("off")

        axes[0, 1].imshow(before_mask, cmap="Greens")
        axes[0, 1].set_title("Before Vegetation Mask")
        axes[0, 1].axis("off")

        axes[0, 2].imshow(after_rgb)
        axes[0, 2].set_title("After RGB")
        axes[0, 2].axis("off")

        axes[1, 0].imshow(after_mask, cmap="Greens")
        axes[1, 0].set_title("After Vegetation Mask")
        axes[1, 0].axis("off")

        axes[1, 1].imshow(raw_deforestation, cmap="Reds")
        axes[1, 1].set_title("Raw Deforestation")
        axes[1, 1].axis("off")

        axes[1, 2].imshow(filtered_mask, cmap="Reds")
        axes[1, 2].set_title("Cleaned Deforestation")
        axes[1, 2].axis("off")

        plt.tight_layout()
        plt.show()
    """

    return before_mask, after_mask, raw_deforestation, filtered_mask

In [6]:
def mask_to_polygons(reference_tif_path, filtered_mask):
    """
    Convert binary mask to polygons using raster georeferencing.
    Returns a GeoDataFrame.
    """

    with rasterio.open(reference_tif_path) as src:
        transform = src.transform
        crs = src.crs

    polygons = []

    for geom, value in shapes(
        filtered_mask.astype("uint8"),
        mask=filtered_mask.astype(bool),
        transform=transform
    ):
        if value == 1:
            polygons.append(shape(geom))

    gdf = gpd.GeoDataFrame(geometry=polygons, crs=crs)

    return gdf

def insert_deforestation_to_postgis(deforestation_gdf, detected_at):
    """
    Insert deforestation polygons into existing PostGIS table:
    detected_deforestation(geometry, detected_at)
    """

    # 1. Convert CRS to EPSG:4326 because your table expects that
    gdf = deforestation_gdf.to_crs(epsg=4326).copy()

    # 2. Add detected date
    gdf["detected_at"] = detected_at

    # 3. Keep only columns that exist in your table
    gdf = gdf[["geometry", "detected_at"]]

    # 4. PostgreSQL connection
    engine = create_engine(
    "postgresql+psycopg2://rajlaxmiawatade@localhost:5432/urban_green_db"
    )

    # 5. Insert into PostGIS table
    gdf.to_postgis(
        name="detected_deforestation",
        con=engine,
        if_exists="append",
        index=False
    )

    print(f"Inserted {len(gdf)} polygons into detected_deforestation")

In [12]:
import psycopg2

def verify_deforestation_against_permits():
    conn = psycopg2.connect(
        dbname="urban_green_db",
        host="localhost",
        port="5432"
    )
    cur = conn.cursor()

    query = """
    DROP TABLE IF EXISTS verification_results;

    CREATE TABLE verification_results AS
    WITH permit_matches AS (
        SELECT
            d.detection_id,
            d.detected_at,
            d.geometry,
            d.centroid,
            p.permit_id,

            -- Areas
            ST_Area(d.geometry::geography) AS detected_area,
            ST_Area(ST_Intersection(d.geometry, p.geometry)::geography) AS intersection_area,

            -- Coverage
            ST_Area(ST_Intersection(d.geometry, p.geometry)::geography)
            / NULLIF(ST_Area(d.geometry::geography), 0) AS coverage_ratio,

            ROW_NUMBER() OVER (
                PARTITION BY d.detection_id
                ORDER BY ST_Area(ST_Intersection(d.geometry, p.geometry)::geography) DESC
            ) AS rn

        FROM detected_deforestation d
        LEFT JOIN permits p
            ON ST_Intersects(d.geometry, p.geometry)
            AND p.status = 'active'
            AND d.detected_at >= p.issue_date
    ),

    best_matches AS (
        SELECT *
        FROM permit_matches
        WHERE rn = 1
    )

    SELECT
        detection_id,
        detected_at,
        permit_id,

        ROUND(detected_area::numeric, 2) AS area_m2,
        ROUND(COALESCE(intersection_area, 0)::numeric, 2) AS intersection_area_m2,
        ROUND(COALESCE(coverage_ratio, 0)::numeric, 3) AS coverage_ratio,

        CASE
            WHEN COALESCE(coverage_ratio, 0) >= 0.8 THEN 'PERMITTED'
            ELSE 'POTENTIALLY_ILLEGAL'
        END AS status,

        centroid,
        geometry

    FROM best_matches;
    """

    cur.execute(query)
    conn.commit()

    cur.close()
    conn.close()

    print("Verification completed. Results stored in verification_results table.")

In [ ]:
import psycopg2

def calculate_summary_stats():
    conn = psycopg2.connect(
        dbname="urban_green_db",
        host="localhost",
        port="5432"
    )
    cur = conn.cursor()

    query = """
    SELECT
    ROUND(COALESCE(SUM(area_m2), 0) / 1000000.0, 4) AS total_green_cover_loss_km2,

    ROUND(COALESCE(SUM(CASE 
        WHEN status = 'POTENTIALLY_ILLEGAL' THEN area_m2 
        ELSE 0 
    END), 0) / 1000000.0, 4) AS illegal_area_km2,

    ROUND(COALESCE(SUM(CASE 
        WHEN status = 'PERMITTED' THEN area_m2 
        ELSE 0 
    END), 0) / 1000000.0, 4) AS legal_area_km2,

    COUNT(CASE 
        WHEN status = 'PERMITTED' THEN 1 
    END) AS total_legal_deforestation_count

    FROM verification_results;
    """

    cur.execute(query)
    result = cur.fetchone()

    stats = {
        "total_green_cover_loss_km2": result[0] or 0,
        "illegal_area_km2": result[1] or 0,
        "legal_area_km2": result[2] or 0,
        "total_legal_deforestation_count": result[3] or 0
    }

    cur.close()
    conn.close()

    return stats

In [ ]:
import joblib
from datetime import date

def analyze(before_tif, after_tif):

    # LOADING RF MODEL
    rf_model = joblib.load("threefeat_new_simple_new_rf_urban_green_model_med.pkl")

    # DEFORESTATION DETECTION AS A MASK
    before_mask, after_mask, _, filtered_mask = detect_deforestation(
    before_path=before_tif,
    after_path=after_tif,
    model=rf_model,
    min_patch_pixels=0    
    )

    # SAVING DEFORESTATION MASK AS POLYGONS IN DB
    deforestation_gdf = mask_to_polygons(before_tif, filtered_mask)
    insert_deforestation_to_postgis(deforestation_gdf, date.today())
    
    # VERIFYING DEFORESTATION AGAINST PERMITS
    verify_deforestation_against_permits()
    stats = calculate_summary_stats()

    return_obj = {
        "before_veg_mask_tif": before_mask,
        "after_veg_mask_tif": after_mask,
        "def_mask": filtered_mask,
        "total_green_cover_loss_km2": stats[0],
        "illegal_area_km2": stats[1],
        "legal_area_km2": stats[2],
        "total_legal_deforestation_count": stats[3]
    }

    return return_obj

    

In [4]:
tif1 = '/Users/rajlaxmiawatade/Desktop/satellite images/delhi/beforepermit.tif'
tif2 = '/Users/rajlaxmiawatade/Desktop/satellite images/delhi/afterpermit.tif'
with rasterio.open(tif1) as src1, rasterio.open(tif2) as src2:
    print("CRS match:", src1.crs == src2.crs)
    print("Transform match:", src1.transform == src2.transform)
    print("Resolution match:", src1.res == src2.res)
    print("Shape match:", src1.shape == src2.shape)

CRS match: True
Transform match: True
Resolution match: True
Shape match: True


In [19]:
analyze(tif1, tif2)

Inserted 301 polygons into detected_deforestation
Verification completed. Results stored in verification_results table.
{'total_green_cover_loss_m2': Decimal('33140.25'), 'illegal_area_m2': Decimal('32138.46'), 'total_legal_deforestation_count': 8}
